In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score


# ==========================================
# 1. LOAD TRAINED CHICAGO ARTIFACTS
# ==========================================
load_path = "E:/NUS/IT5006/Crime Prediction Project/Saved_Models/"

loaded_models = joblib.load(os.path.join(load_path, 'crime_independent_models.joblib'))
loaded_scaler = joblib.load(os.path.join(load_path, 'data_scaler.joblib'))
trained_features = joblib.load(os.path.join(load_path, 'feature_names.joblib'))

print("✅ Independent models, scaler, and features loaded.")


# ==========================================
# 2. NIBRS SEVERITY MAPPING
# ==========================================
nibrs_severity_mapping = {
    # TIER 1 - LETHAL
    '09A': 'Tier 1 - Lethal',
    '09B': 'Tier 1 - Lethal',
    '11A': 'Tier 1 - Lethal',
    '100': 'Tier 1 - Lethal',
    '64A': 'Tier 1 - Lethal',
    '64B': 'Tier 1 - Lethal',

    # TIER 2 - PERSONAL VIOLENCE
    '120': 'Tier 2 - Personal Violence',
    '13A': 'Tier 2 - Personal Violence',
    '13B': 'Tier 2 - Personal Violence',
    '11B': 'Tier 2 - Personal Violence',
    '11C': 'Tier 2 - Personal Violence',
    '11D': 'Tier 2 - Personal Violence',
    '36A': 'Tier 2 - Personal Violence',
    '36B': 'Tier 2 - Personal Violence',

    # TIER 3 - PROPERTY
    '220': 'Tier 3 - Property',
    '23A': 'Tier 3 - Property',
    '23C': 'Tier 3 - Property',
    '23H': 'Tier 3 - Property',
    '240': 'Tier 3 - Property',
    '200': 'Tier 3 - Property',
    '290': 'Tier 3 - Property',
    '13C': 'Tier 3 - Property',
    '26A': 'Tier 3 - Property',
    '90J': 'Tier 3 - Property',

    # TIER 4 - PUBLIC ORDER
    '35A': 'Tier 4 - Public Order',
    '35B': 'Tier 4 - Public Order',
    '40A': 'Tier 4 - Public Order',
    '520': 'Tier 4 - Public Order',
    '39A': 'Tier 4 - Public Order',
    '90C': 'Tier 4 - Public Order',
    '90D': 'Tier 4 - Public Order',
    '90G': 'Tier 4 - Public Order',
}

tiers = [
    'Tier 1 - Lethal',
    'Tier 2 - Personal Violence',
    'Tier 3 - Property',
    'Tier 4 - Public Order'
]

target_cols = ['y_tier1', 'y_tier2', 'y_tier3', 'y_tier4']

✅ Independent models, scaler, and features loaded.


In [ ]:
# ==========================================
# 3. LOAD AND MERGE NIBRS DATA
# ==========================================


offenses = pd.read_csv(
    "E:/NUS/IT5006/Crime Prediction Project/Crime Data/Illinois 2024/NIBRS_OFFENSE.csv"
)
incidents = pd.read_csv(
    "E:/NUS/IT5006/Crime Prediction Project/Crime Data/Illinois 2024/NIBRS_incident.csv"
)

df_nibrs = offenses.merge(
    incidents[['incident_id', 'incident_date', 'agency_id']],
    on='incident_id',
    how='inner'
)

df_nibrs['incident_date'] = pd.to_datetime(df_nibrs['incident_date'], errors='coerce')
df_nibrs = df_nibrs.dropna(subset=['incident_date', 'agency_id', 'offense_code'])

df_nibrs['Severity_Tier'] = df_nibrs['offense_code'].map(nibrs_severity_mapping).fillna('Other')

print("✅ Merge complete.")
print(df_nibrs[['offense_code', 'Severity_Tier']].head())

Loading NIBRS files...


C:\Users\tiany\AppData\Local\Temp\ipykernel_158672\1673759747.py:9: DtypeWarning: Columns (0: cleared_except_date) have mixed types. Specify dtype option on import or set low_memory=False.
  incidents = pd.read_csv(


✅ Merge complete.
  offense_code               Severity_Tier
0          13B  Tier 2 - Personal Violence
1          13C           Tier 3 - Property
2          13B  Tier 2 - Personal Violence
3          290           Tier 3 - Property
4          26A           Tier 3 - Property


In [ ]:
# ==========================================
# 4. BUILD DAILY AGENCY-LEVEL PANEL
# ==========================================
# Normalize to daily level
df_nibrs['Date'] = df_nibrs['incident_date'].dt.normalize()

# Keep only mapped tiers if you want evaluation focused on your 4 classes
df_nibrs_eval = df_nibrs[df_nibrs['Severity_Tier'].isin(tiers)].copy()

# Aggregate counts per agency-day-tier
df_daily_nibrs = (
    df_nibrs_eval.groupby(['agency_id', 'Date', 'Severity_Tier'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure all tier columns exist
for tier in tiers:
    if tier not in df_daily_nibrs.columns:
        df_daily_nibrs[tier] = 0

# Sort before lag creation
df_daily_nibrs = df_daily_nibrs.sort_values(['agency_id', 'Date']).reset_index(drop=True)

print(f"Daily NIBRS panel created. Shape: {df_daily_nibrs.shape}")

✅ Daily NIBRS panel created. Shape: (92729, 6)


In [ ]:
# ==========================================
# 5. CREATE LAG FEATURES
# ==========================================
for i, tier in enumerate(tiers):
    df_daily_nibrs[f"tier{i+1}_lag_7d"] = (
        df_daily_nibrs.groupby('agency_id')[tier]
        .transform(lambda x: x.shift(1).rolling(7, min_periods=1).sum())
    )
    df_daily_nibrs[f"tier{i+1}_lag_30d"] = (
        df_daily_nibrs.groupby('agency_id')[tier]
        .transform(lambda x: x.shift(1).rolling(30, min_periods=1).sum())
    )

# Fill lag NaNs
df_daily_nibrs = df_daily_nibrs.fillna(0)

In [ ]:
# ==========================================
# 6. CREATE CYCLICAL TIME FEATURES
# ==========================================
df_daily_nibrs['month'] = df_daily_nibrs['Date'].dt.month
df_daily_nibrs['month_sin'] = np.sin(2 * np.pi * df_daily_nibrs['month'] / 12)
df_daily_nibrs['month_cos'] = np.cos(2 * np.pi * df_daily_nibrs['month'] / 12)

df_daily_nibrs['dow'] = df_daily_nibrs['Date'].dt.dayofweek
df_daily_nibrs['dow_sin'] = np.sin(2 * np.pi * df_daily_nibrs['dow'] / 7)
df_daily_nibrs['dow_cos'] = np.cos(2 * np.pi * df_daily_nibrs['dow'] / 7)

df_daily_nibrs = df_daily_nibrs.drop(columns=['month', 'dow'])

In [ ]:
# ==========================================
# 7. CREATE TARGETS
# ==========================================
df_daily_nibrs['y_tier1'] = (df_daily_nibrs['Tier 1 - Lethal'] > 0).astype(int)
df_daily_nibrs['y_tier2'] = (df_daily_nibrs['Tier 2 - Personal Violence'] > 0).astype(int)
df_daily_nibrs['y_tier3'] = (df_daily_nibrs['Tier 3 - Property'] > 0).astype(int)
df_daily_nibrs['y_tier4'] = (df_daily_nibrs['Tier 4 - Public Order'] > 0).astype(int)

y_nibrs_actual = df_daily_nibrs[target_cols].copy()

In [ ]:
# ==========================================
# 8. ALIGN FEATURES WITH CHICAGO TRAINING FEATURES
# ==========================================
for col in trained_features:
    if col not in df_daily_nibrs.columns:
        df_daily_nibrs[col] = 0

X_nibrs = df_daily_nibrs[trained_features].copy()
X_nibrs_scaled = loaded_scaler.transform(X_nibrs)

print(f" NIBRS feature matrix ready. Shape: {X_nibrs.shape}")

✅ NIBRS feature matrix ready. Shape: (92729, 12)


In [ ]:
# ==========================================
# 9. DEFINE WHICH MODELS NEED SCALING
# ==========================================
model_scaling_map = {
    'Logistic Regression': True,
    'Random Forest': False,
    'XGBoost': False,
    'Neural Network': True
}

# Use same thresholds as Chicago evaluation
thresholds = [0.10, 0.20, 0.35, 0.35]

In [ ]:
# ==========================================
# 10. RUN EACH MODEL SEPARATELY ON NIBRS
# ==========================================
all_results = []
all_probabilities = {}



for model_name, estimators in loaded_models.items():
    print(f"\n--- Predicting with {model_name} ---")

    use_scaled = model_scaling_map.get(model_name, False)
    X_input = X_nibrs_scaled if use_scaled else X_nibrs

    prob_matrix = []
    pred_matrix = []

    for i, target_col in enumerate(target_cols):
        clf = estimators[i]
        probs = clf.predict_proba(X_input)[:, 1]
        preds = (probs >= thresholds[i]).astype(int)

        prob_matrix.append(probs)
        pred_matrix.append(preds)

    prob_matrix = np.array(prob_matrix).T
    pred_matrix = np.array(pred_matrix).T

    all_probabilities[model_name] = prob_matrix

    for i, target_col in enumerate(target_cols):
        y_true = y_nibrs_actual[target_col].values
        y_prob = prob_matrix[:, i]
        y_pred = pred_matrix[:, i]

        try:
            auc = roc_auc_score(y_true, y_prob)
        except ValueError:
            auc = np.nan

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        all_results.append({
            'Model': model_name,
            'Crime Tier': target_col,
            'Precision': round(precision, 4),
            'Recall': round(recall, 4),
            'F1-Score': round(f1, 4),
            'AUC-ROC': round(auc, 4) if not np.isnan(auc) else np.nan
        })

results_df = pd.DataFrame(all_results)

print("\n=== NIBRS GENERALIZATION RESULTS (SEPARATE MODELS) ===")
print(results_df.to_string(index=False))


=== RUNNING SEPARATE MODEL TESTING ON NIBRS ===

--- Predicting with Logistic Regression ---

--- Predicting with Random Forest ---

--- Predicting with XGBoost ---

--- Predicting with Neural Network ---

=== NIBRS GENERALIZATION RESULTS (SEPARATE MODELS) ===
              Model Crime Tier  Precision  Recall  F1-Score  AUC-ROC
Logistic Regression    y_tier1     0.2211  0.3931    0.2830   0.7329
Logistic Regression    y_tier2     0.4764  0.9991    0.6452   0.7222
Logistic Regression    y_tier3     0.7795  0.9965    0.8748   0.7166
Logistic Regression    y_tier4     0.5617  0.4531    0.5016   0.7483
      Random Forest    y_tier1     0.0419  0.0162    0.0234   0.6134
      Random Forest    y_tier2     0.4894  0.9697    0.6505   0.6176
      Random Forest    y_tier3     0.7818  0.9809    0.8701   0.6760
      Random Forest    y_tier4     0.3904  0.1380    0.2039   0.6090
            XGBoost    y_tier1     0.0327  0.2711    0.0583   0.3244
            XGBoost    y_tier2     0.4600  0.934

In [ ]:
# ==========================================
# 11. SPATIAL / JURISDICTIONAL EVALUATION FOR TIER 2
# ==========================================


spatial_results = []

total_agency_days = len(df_daily_nibrs)
total_actual_tier2 = df_daily_nibrs['y_tier2'].sum()
base_rate = total_actual_tier2 / total_agency_days if total_agency_days > 0 else 0

for model_name, prob_matrix in all_probabilities.items():
    preds_tier2 = (prob_matrix[:, 1] >= thresholds[1]).astype(int)

    hits = ((preds_tier2 == 1) & (df_daily_nibrs['y_tier2'].values == 1)).sum()
    total_flags = (preds_tier2 == 1).sum()

    spatial_precision = hits / total_flags if total_flags > 0 else 0
    lift = spatial_precision / base_rate if base_rate > 0 else 0

    area_flagged_rate = total_flags / total_agency_days if total_agency_days > 0 else 0
    hit_rate = hits / total_actual_tier2 if total_actual_tier2 > 0 else 0
    pai = hit_rate / area_flagged_rate if area_flagged_rate > 0 else 0

    spatial_results.append({
        'Model': model_name,
        'Tier': 'y_tier2',
        'Base Rate': round(base_rate, 4),
        'Spatial Precision': round(spatial_precision, 4),
        'Lift': round(lift, 4),
        'PAI': round(pai, 4),
        'Days Flagged': int(total_flags)
    })

spatial_results_df = pd.DataFrame(spatial_results)

print(spatial_results_df.to_string(index=False))


=== NIBRS JURISDICTIONAL EVALUATION (TIER 2) ===
              Model    Tier  Base Rate  Spatial Precision   Lift    PAI  Days Flagged
Logistic Regression y_tier2     0.4754             0.4764 1.0021 1.0021         92445
      Random Forest y_tier2     0.4754             0.4894 1.0295 1.0295         87338
            XGBoost y_tier2     0.4754             0.4600 0.9677 0.9677         89541
     Neural Network y_tier2     0.4754             0.4833 1.0167 1.0167         90094


In [ ]:
# ==========================================
# 12. OPTIONAL: THRESHOLD CHECK FOR TIER 2 BY MODEL
# ==========================================


threshold_grid = [0.25, 0.35, 0.45, 0.55, 0.65]

for model_name, prob_matrix in all_probabilities.items():
    print(f"\n--- {model_name} ---")
    print(f"{'Threshold':<10} | {'Precision':<10} | {'Lift':<10} | {'Days Flagged'}")
    print("-" * 55)

    for t in threshold_grid:
        preds = (prob_matrix[:, 1] >= t).astype(int)
        flags = preds.sum()

        if flags > 0:
            hits = ((preds == 1) & (df_daily_nibrs['y_tier2'].values == 1)).sum()
            precision = hits / flags
            lift = precision / base_rate if base_rate > 0 else 0
        else:
            precision = 0
            lift = 0

        print(f"{t:<10.2f} | {precision:<10.2%} | {lift:<10.2f}x | {flags}")


=== THRESHOLD SENSITIVITY (TIER 2) ===

--- Logistic Regression ---
Threshold  | Precision  | Lift       | Days Flagged
-------------------------------------------------------
0.25       | 49.88%     | 1.05      x | 84525
0.35       | 57.46%     | 1.21      x | 60729
0.45       | 67.01%     | 1.41      x | 38970
0.55       | 74.72%     | 1.57      x | 26513
0.65       | 81.07%     | 1.71      x | 18494

--- Random Forest ---
Threshold  | Precision  | Lift       | Days Flagged
-------------------------------------------------------
0.25       | 49.86%     | 1.05      x | 83537
0.35       | 53.30%     | 1.12      x | 68765
0.45       | 56.44%     | 1.19      x | 42637
0.55       | 58.36%     | 1.23      x | 17280
0.65       | 58.72%     | 1.24      x | 4402

--- XGBoost ---
Threshold  | Precision  | Lift       | Days Flagged
-------------------------------------------------------
0.25       | 45.90%     | 0.97      x | 89262
0.35       | 45.68%     | 0.96      x | 88671
0.45       | 45.

In [ ]:
# ==========================================
# 13. TEMPORAL ACCURACY (WEEKDAY VS WEEKEND)
# ==========================================


# Create the 'is_weekend' flag using your actual dataframe: df_daily_nibrs
df_daily_nibrs['is_weekend'] = df_daily_nibrs['Date'].dt.dayofweek >= 5

for model_name, prob_matrix in all_probabilities.items():
    # Using 0.20 threshold for Tier 2 (matching your Chicago evaluation)
    pred_tier2 = (prob_matrix[:, 1] >= 0.20).astype(int)

    # Filter for weekends and weekdays
    weekend_mask = df_daily_nibrs['is_weekend']
    weekday_mask = ~df_daily_nibrs['is_weekend']

    # Calculate accuracy
    weekend_acc = (pred_tier2[weekend_mask] == df_daily_nibrs['y_tier2'][weekend_mask]).mean()
    weekday_acc = (pred_tier2[weekday_mask] == df_daily_nibrs['y_tier2'][weekday_mask]).mean()

    print(f"{model_name:>20} - Weekday Acc: {weekday_acc:.2%} | Weekend Acc: {weekend_acc:.2%}")


=== TEMPORAL ACCURACY (WEEKDAY VS WEEKEND) ===
 Logistic Regression - Weekday Acc: 45.90% | Weekend Acc: 52.63%
       Random Forest - Weekday Acc: 49.04% | Weekend Acc: 54.22%
             XGBoost - Weekday Acc: 42.84% | Weekend Acc: 49.76%
      Neural Network - Weekday Acc: 47.77% | Weekend Acc: 53.01%
